# Churn Labeling

1. **Easy Rule:** >365 Tage keine Bestellung = churned (1)
2. **Advanced:** 2016-2020 bestellt, 2020-2025 nie wieder = churned (1)

In [ ]:
import pandas as pd

panel = pd.read_csv('Processed Data/big_panel.csv', parse_dates=['ORDER_DATE'])
orders_only = panel[panel['is_return_order'] == False].copy()

## Easy Rule: 1 Jahr Inaktivitaet

In [ ]:
last_order = orders_only.groupby('H_ID')['ORDER_DATE'].max().reset_index()
last_order.columns = ['H_ID', 'last_order_date']

cutoff = panel['ORDER_DATE'].max() - pd.Timedelta(days=365)
last_order['churn_easy'] = (last_order['last_order_date'] < cutoff).astype(int)

## Advanced: Zeitbasierter Split

In [ ]:
train_hids = set(orders_only[
    (orders_only['ORDER_DATE'] >= '2016-01-01') &
    (orders_only['ORDER_DATE'] <= '2020-12-31')
]['H_ID'].unique())

val_hids = set(orders_only[
    (orders_only['ORDER_DATE'] >= '2020-01-01') &
    (orders_only['ORDER_DATE'] <= '2025-12-31')
]['H_ID'].unique())

churned_hids = train_hids - val_hids

last_order['churn_advanced'] = pd.NA
last_order.loc[last_order['H_ID'].isin(train_hids - churned_hids), 'churn_advanced'] = 0
last_order.loc[last_order['H_ID'].isin(churned_hids), 'churn_advanced'] = 1

## Merge churn labels onto big_panel and save

In [ ]:
churn_map = last_order[['H_ID', 'churn_easy', 'churn_advanced']]
panel = panel.merge(churn_map, on='H_ID', how='left')

panel.to_csv('Processed Data/big_panel.csv', index=False)

## Overlap between Easy and Advanced

In [ ]:
compare = last_order.dropna(subset=['churn_advanced']).copy()
compare['churn_advanced'] = compare['churn_advanced'].astype(int)

easy_churned = (compare['churn_easy'] == 1).sum()
adv_churned = (compare['churn_advanced'] == 1).sum()
both_churned = ((compare['churn_easy'] == 1) & (compare['churn_advanced'] == 1)).sum()
neither = ((compare['churn_easy'] == 0) & (compare['churn_advanced'] == 0)).sum()
easy_only = ((compare['churn_easy'] == 1) & (compare['churn_advanced'] == 0)).sum()
adv_only = ((compare['churn_easy'] == 0) & (compare['churn_advanced'] == 1)).sum()
total = len(compare)

print(f'Total customers with advanced label: {total:,}')
print(f'')
print(f'churn_easy = 1:        {easy_churned:>8,} ({easy_churned/total*100:.1f}%)')
print(f'churn_advanced = 1:    {adv_churned:>8,} ({adv_churned/total*100:.1f}%)')
print(f'')
print(f'Both churned:          {both_churned:>8,} ({both_churned/total*100:.1f}%)')
print(f'Easy only:             {easy_only:>8,} ({easy_only/total*100:.1f}%)')
print(f'Advanced only:         {adv_only:>8,} ({adv_only/total*100:.1f}%)')
print(f'Neither:               {neither:>8,} ({neither/total*100:.1f}%)')
print(f'')
print(f'Of churn_easy=1, {both_churned/easy_churned*100:.1f}% are also churn_advanced=1')
print(f'Of churn_advanced=1, {both_churned/adv_churned*100:.1f}% are also churn_easy=1')